# SURENA Cartesian Motion Benchmarks

How accurately can the SURENA arm execute predefined Cartesian and joint trajectories?


## 1. Environment setup

In [ ]:
from __future__ import annotations

import json
import os
import time
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from mpl_toolkits.mplot3d import Axes3D
import mujoco
import numpy as np

from surena_vla.control import (
    ARM_JOINT_NAMES_BARE,
    HOME_QPOS,
    JOINT_LIMITS,
    SurenaArmController,
    clamp_joints,
)
from surena_vla.paths import OUTPUTS_ROOT, SURENA_ARM_XML


# Simulation and sampling conditions.
SIM_FREQ_HZ = 500.0
CTRL_FREQ_HZ = 20.0
STEPS_PER_CTRL = int(round(SIM_FREQ_HZ / CTRL_FREQ_HZ))
CTRL_TICKS_PER_CARTESIAN_WAYPOINT = 1
CTRL_TICKS_PER_JOINT_WAYPOINT = 1
SETTLE_SECONDS = 0.5
FIGURE_DPI = 320

# Select one figure style. Options: "thesis", "presentation", "compact", "diagnostic".
FIGURE_STYLE_MODE = "thesis"
FIGURE_STYLE_PRESETS = {
    "thesis": {
        "dpi": 320, "font_size": 10, "title_size": 11, "legend_size": 9,
        "line_width": 1.9, "reference_line_width": 1.7,
        "tick_width": 0.8, "tick_length": 3.5, "grid_alpha": 1.0,
        "single_size": (7.2, 4.6), "trajectory_size": (8.0, 6.6),
        "dashboard_size": (13.6, 7.0),
    },
    "presentation": {
        "dpi": 180, "font_size": 13, "title_size": 16, "legend_size": 11,
        "line_width": 3.0, "reference_line_width": 3.2,
        "tick_width": 1.5, "tick_length": 6.0, "grid_alpha": 0.35,
        "single_size": (11, 6.5), "trajectory_size": (9, 8), "dashboard_size": (19, 12),
    },
    "compact": {
        "dpi": 200, "font_size": 9, "title_size": 11, "legend_size": 8,
        "line_width": 1.8, "reference_line_width": 2.0,
        "tick_width": 1.0, "tick_length": 4.0, "grid_alpha": 0.25,
        "single_size": (7, 4.2), "trajectory_size": (6.5, 5.8), "dashboard_size": (13, 8),
    },
    "diagnostic": {
        "dpi": 180, "font_size": 10, "title_size": 12, "legend_size": 8,
        "line_width": 2.0, "reference_line_width": 2.2,
        "tick_width": 1.1, "tick_length": 5.0, "grid_alpha": 0.45,
        "single_size": (10, 5.5), "trajectory_size": (8, 7), "dashboard_size": (17, 10),
    },
}

# Interactive display selector. Use "all", one name, or a tuple of names.
# Examples: "circle" or ("circle", "square", "small_square").
DISPLAY_BENCHMARK_MODES = "all"
STATE_LOGGING_FREQUENCY_HZ = CTRL_FREQ_HZ
COMMAND_REGIME = "step_input_one_control_tick_per_waypoint"
CARTESIAN_ORIENTATION_POLICY = "fixed_initial_eef_orientation_full_pose_constraint"

# Frozen controller configuration used for every benchmark and repeatability trial.
CONTROLLER_CONFIG = {
    "kp_major": 1200.0,
    "kp_wrist": 600.0,
    "damping_major": 45.0,
    "damping_wrist": 25.0,
    "gravity_strength": 1.0,
    "actuator_force_limits": "disabled_by_tune_tracking_default",
}

# Fixed benchmark geometry.
BENCHMARK_GEOMETRY = {
    "straight_delta_m": np.array([0.00, 0.10, 0.00]),
    "diagonal_delta_m": np.array([-0.04, 0.06, 0.05]),
    "circle_radius_m": 0.05,
    "square_side_m": 0.08,
    "small_square_side_m": 0.04,
    "line_points": 41,
    "circle_points": 81,
    "square_points_per_side": 20,
    "small_square_points_per_side": 11,
    "joint_points_per_segment": 30,
    "elbow_flex_rad": -0.90,
    "elbow_extend_rad": -0.20,
    "shoulder_pitch_first_rad": -0.60,
    "shoulder_pitch_second_rad": 0.10,
    "wrist_pitch_first_rad": -0.45,
    "wrist_pitch_second_rad": 0.25,
}

REPEATABILITY_TRIALS = 5
REPEATABILITY_GEOMETRIES = ("straight_line", "circle")
BENCHMARK_OUTPUT_ROOT = Path(
    os.environ.get(
        "SURENA_CARTESIAN_BENCHMARK_OUTPUTS",
        OUTPUTS_ROOT / "cartesian_motion_benchmarks",
    )
).expanduser().resolve()

# Match the typography, palette, and export treatment of surena_thesis_evaluation.ipynb.
PAPER_RC = {
    "figure.dpi": 120,
    "savefig.dpi": 320,
    "font.size": 12,
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif", "Liberation Serif"],
    "mathtext.fontset": "dejavuserif",
    "axes.facecolor": "white",
    "axes.grid": True,
    "grid.color": ".8",
    "axes.edgecolor": "0.15",
    "axes.linewidth": 0.8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "legend.frameon": True,
    "legend.framealpha": 0.9,
    "legend.edgecolor": "#cccccc",
}
C_BLUE, C_RED, C_GREEN = "#4477AA", "#EE6677", "#228833"
C_YELLOW, C_CYAN, C_PURPLE, C_GREY = "#CCBB44", "#66CCEE", "#AA3377", "#505050"
C_ORANGE = "#F59F00"
plt.rcParams.update(PAPER_RC)

print(f"Simulation: {SIM_FREQ_HZ:.0f} Hz")
print(f"Controller: {CTRL_FREQ_HZ:.0f} Hz ({STEPS_PER_CTRL} physics steps/tick)")
print("Output root:", BENCHMARK_OUTPUT_ROOT)


## 2. Load the SURENA MuJoCo model

In [ ]:
model = mujoco.MjModel.from_xml_path(str(SURENA_ARM_XML))
model.opt.timestep = 1.0 / SIM_FREQ_HZ
data = mujoco.MjData(model)
mujoco.mj_forward(model, data)

print("MJCF:", SURENA_ARM_XML)
print(f"nq={model.nq}, nv={model.nv}, nu={model.nu}")
print(f"MuJoCo timestep={model.opt.timestep:.6f} s")

## 3. Initialize the controller


In [ ]:
def _physics_step(controller: SurenaArmController) -> None:
    '''Advance one MuJoCo step using the frozen gravity-compensation setting.'''
    if controller._gravity_comp_enabled:
        mujoco.mj_forward(model, data)
        controller.apply_gravity_compensation()
    mujoco.mj_step(model, data)
    data.qfrc_applied[:] = 0.0


def settle_controller(controller: SurenaArmController, seconds: float = SETTLE_SECONDS) -> None:
    n_steps = max(1, int(round(float(seconds) * SIM_FREQ_HZ)))
    controller.bridge.control_callback()
    for physics_step in range(n_steps):
        if physics_step % STEPS_PER_CTRL == 0:
            controller.bridge.control_callback()
        _physics_step(controller)
    mujoco.mj_forward(model, data)


def reset_robot(*, verbose: bool = True) -> SurenaArmController:
    mujoco.mj_resetData(model, data)
    mujoco.mj_forward(model, data)

    controller = SurenaArmController(model=model, data=data, prefix="", apply_home=True)
    controller.tune_tracking(
        kp_major=CONTROLLER_CONFIG["kp_major"],
        kp_wrist=CONTROLLER_CONFIG["kp_wrist"],
        damping_major=CONTROLLER_CONFIG["damping_major"],
        damping_wrist=CONTROLLER_CONFIG["damping_wrist"],
        verbose=False,
    )
    controller.enable_gravity_compensation(
        strength=CONTROLLER_CONFIG["gravity_strength"],
        verbose=False,
    )
    settle_controller(controller)
    controller.reset_ik()

    if verbose:
        eef_pos, _ = controller.get_eef_pose()
        print("Controller reset at HOME")
        print("  q [rad]:", np.round(controller.get_arm_qpos(), 4))
        print("  EEF [m]:", np.round(eef_pos, 4))
    return controller


ctrl = reset_robot()


## 4. Reference trajectory generators

In [ ]:
def _as_vector(value, size: int, name: str) -> np.ndarray:
    array = np.asarray(value, dtype=float)
    if array.shape != (size,) or not np.all(np.isfinite(array)):
        raise ValueError(f"{name} must be a finite shape-({size},) vector")
    return array.copy()


def _segment(start: np.ndarray, end: np.ndarray, n_points: int, *, include_start: bool = True) -> np.ndarray:
    if int(n_points) < 2:
        raise ValueError("Each segment needs at least two points")
    values = np.linspace(start, end, int(n_points), dtype=float)
    return values if include_start else values[1:]


def generate_line(
    start,
    *,
    target=None,
    delta=None,
    n_points: int = 41,
    name: str = "straight_line",
) -> dict:
    start = _as_vector(start, 3, "start")
    if (target is None) == (delta is None):
        raise ValueError("Specify exactly one of target or delta")
    end = _as_vector(target, 3, "target") if target is not None else start + _as_vector(delta, 3, "delta")
    return {
        "name": str(name),
        "space": "cartesian",
        "points": _segment(start, end, n_points),
        "metadata": {"start_m": start, "target_m": end, "n_points": int(n_points)},
    }


def generate_circle(
    start,
    *,
    radius: float = 0.05,
    n_points: int = 81,
) -> dict:
    '''Generate one closed YZ circle with x constant and the first point at start.'''
    start = _as_vector(start, 3, "start")
    radius = float(radius)
    if radius <= 0.0 or int(n_points) < 5:
        raise ValueError("Circle radius must be positive and n_points must be at least 5")

    center = start.copy()
    center[1] -= radius
    theta = np.linspace(0.0, 2.0 * np.pi, int(n_points), endpoint=True)
    points = np.column_stack([
        np.full_like(theta, center[0]),
        center[1] + radius * np.cos(theta),
        center[2] + radius * np.sin(theta),
    ])
    return {
        "name": "circle",
        "space": "cartesian",
        "points": points,
        "metadata": {
            "center_m": center,
            "radius_m": radius,
            "plane": "yz",
            "n_points": int(n_points),
        },
    }


def generate_square(
    start,
    *,
    side: float = 0.08,
    points_per_side: int = 20,
    name: str = "square",
) -> dict:
    '''Generate a closed four-segment square in the YZ plane.'''
    start = _as_vector(start, 3, "start")
    side = float(side)
    if side <= 0.0 or int(points_per_side) < 2:
        raise ValueError("Square side must be positive and points_per_side at least 2")

    vertices = np.array([
        start,
        start + np.array([0.0, side, 0.0]),
        start + np.array([0.0, side, side]),
        start + np.array([0.0, 0.0, side]),
        start,
    ])
    segments = [
        _segment(vertices[i], vertices[i + 1], points_per_side, include_start=(i == 0))
        for i in range(4)
    ]
    return {
        "name": str(name),
        "space": "cartesian",
        "points": np.vstack(segments),
        "metadata": {
            "vertices_m": vertices,
            "side_m": side,
            "plane": "yz",
            "points_per_side": int(points_per_side),
        },
    }


def generate_joint_sweep(
    home=HOME_QPOS,
    *,
    joint_name: str,
    first_angle: float,
    second_angle: float,
    points_per_segment: int = 30,
    name=None,
) -> dict:
    """Generate home → first angle → second angle → home for one named joint."""
    home = _as_vector(home, 7, "home")
    if joint_name not in ARM_JOINT_NAMES_BARE:
        raise ValueError(f"Unknown arm joint: {joint_name!r}")
    joint_index = ARM_JOINT_NAMES_BARE.index(joint_name)
    first = home.copy()
    second = home.copy()
    first[joint_index] = float(first_angle)
    second[joint_index] = float(second_angle)
    requested_keyframes = np.vstack([home, first, second, home])
    raw_segments = [
        _segment(requested_keyframes[i], requested_keyframes[i + 1], points_per_segment, include_start=(i == 0))
        for i in range(3)
    ]
    raw_points = np.vstack(raw_segments)
    points = clamp_joints(raw_points)
    generator_clip_events = int(np.sum(np.any(np.abs(points - raw_points) > 1e-12, axis=1)))
    return {
        "name": str(name or f"{joint_name[:-6] if joint_name.endswith('_joint') else joint_name}_motion"),
        "space": "joint",
        "points": points,
        "metadata": {
            "joint_name": joint_name,
            "joint_index": joint_index,
            "requested_keyframes_rad": requested_keyframes,
            "executed_keyframes_rad": clamp_joints(requested_keyframes),
            "points_per_segment": int(points_per_segment),
            "generator_clip_events": generator_clip_events,
        },
    }


def generate_joint_motion(
    home=HOME_QPOS,
    *,
    flex_angle: float = -0.90,
    extend_angle: float = -0.20,
    points_per_segment: int = 30,
) -> dict:
    """Compatibility wrapper for the elbow benchmark."""
    return generate_joint_sweep(
        home,
        joint_name="r_elbow_pitch_joint",
        first_angle=flex_angle,
        second_angle=extend_angle,
        points_per_segment=points_per_segment,
        name="elbow_joint_motion",
    )


In [ ]:
# Generator sanity checks: endpoint closure, scale, and joint coverage.
_test_start = np.array([0.40, -0.30, 1.10])
assert np.allclose(generate_line(_test_start, delta=[0.0, 0.1, 0.0])["points"][0], _test_start)
assert np.allclose(generate_circle(_test_start)["points"][[0, -1]], _test_start)
assert np.allclose(generate_square(_test_start)["points"][[0, -1]], _test_start)
assert np.allclose(generate_joint_motion()["points"][[0, -1]], HOME_QPOS)
for _joint_name, _first, _second in (
    ("r_arm_pitch_joint", -0.60, 0.10),
    ("r_elbow_pitch_joint", -0.90, -0.20),
    ("r_hand_pitch_joint", -0.45, 0.25),
):
    _sweep = generate_joint_sweep(
        joint_name=_joint_name,
        first_angle=_first,
        second_angle=_second,
        points_per_segment=5,
    )
    assert np.allclose(_sweep["points"][[0, -1]], HOME_QPOS)
    assert _sweep["metadata"]["generator_clip_events"] == 0
print("Trajectory generator checks passed.")


## 5. Trajectory executor

In [ ]:
def _smoothstep(value: float) -> float:
    value = float(np.clip(value, 0.0, 1.0))
    return value * value * (3.0 - 2.0 * value)


def _forward_kinematics(controller: SurenaArmController, arm_q) -> np.ndarray:
    scratch = mujoco.MjData(model)
    scratch.qpos[:] = data.qpos
    for qadr, value in zip(controller.ik._qadr, _as_vector(arm_q, 7, "arm_q")):
        scratch.qpos[int(qadr)] = float(value)
    mujoco.mj_forward(model, scratch)
    return scratch.site_xpos[controller.ik._site_id].copy()


def _run_control_tick(controller: SurenaArmController, q_command) -> None:
    controller.bridge.publish_arm_qpos(clamp_joints(np.asarray(q_command, dtype=float)))
    controller.bridge.control_callback()
    for _ in range(STEPS_PER_CTRL):
        _physics_step(controller)


def execute_cartesian_trajectory(
    reference_trajectory,
    *,
    controller=None,
    ctrl_ticks_per_waypoint: int = CTRL_TICKS_PER_CARTESIAN_WAYPOINT,
    verbose: bool = True,
    show_every: int = 10,
) -> dict:
    '''Execute a fixed Cartesian target sequence and return a synchronized log.'''
    controller = ctrl if controller is None else controller
    if isinstance(reference_trajectory, dict):
        trajectory = reference_trajectory
        reference = np.asarray(trajectory["points"], dtype=float)
        name = str(trajectory.get("name", "cartesian_trajectory"))
        metadata = dict(trajectory.get("metadata", {}))
    else:
        reference = np.asarray(reference_trajectory, dtype=float)
        name = "cartesian_trajectory"
        metadata = {}
    if reference.ndim != 2 or reference.shape[1] != 3 or len(reference) < 2:
        raise ValueError("reference_trajectory must contain at least two finite XYZ points")
    if not np.all(np.isfinite(reference)):
        raise ValueError("reference_trajectory contains non-finite values")
    if int(ctrl_ticks_per_waypoint) < 1:
        raise ValueError("ctrl_ticks_per_waypoint must be positive")

    fixed_orientation = controller.current_eef_so3()
    initial_eef, initial_quat = controller.get_eef_pose()
    initial_q = controller.get_arm_qpos()

    reference_dense = [reference[0].copy()]
    actual_eef = [initial_eef.copy()]
    actual_joints = [initial_q.copy()]
    commanded_joints = [initial_q.copy()]
    timestamps = [0.0]
    ik_status = []
    ik_position_error = []
    ik_rotation_error = []
    ik_ok = []
    ik_accepted = []
    ik_solve_time_ms = []
    actual_at_waypoints = [initial_eef.copy()]
    joint_at_waypoints = [initial_q.copy()]

    wall_start = time.perf_counter()
    control_tick = 0
    joint_command_clip_events = 0

    for waypoint_index, target in enumerate(reference[1:], start=1):
        ik_info = controller.move_eef_to(target, fixed_orientation, verbose=False)
        q_start = controller.get_arm_qpos().copy()
        q_goal = np.asarray(ik_info["q_goal"], dtype=float).copy()

        ik_status.append(str(ik_info["status"]))
        ik_position_error.append(float(ik_info["pos_err"]))
        ik_rotation_error.append(float(ik_info["rot_err"]))
        ik_ok.append(bool(ik_info["ok"]))
        ik_accepted.append(bool(ik_info["accepted"]))
        ik_solve_time_ms.append(float(ik_info["solve_time_ms"]))

        previous_target = reference[waypoint_index - 1]
        for local_tick in range(int(ctrl_ticks_per_waypoint)):
            alpha = _smoothstep((local_tick + 1) / int(ctrl_ticks_per_waypoint))
            q_command_raw = (1.0 - alpha) * q_start + alpha * q_goal
            q_command = clamp_joints(q_command_raw)
            joint_command_clip_events += int(np.any(np.abs(q_command - q_command_raw) > 1e-12))
            _run_control_tick(controller, q_command)
            control_tick += 1

            eef_now, _ = controller.get_eef_pose()
            reference_now = (1.0 - alpha) * previous_target + alpha * target
            reference_dense.append(reference_now.copy())
            actual_eef.append(eef_now.copy())
            actual_joints.append(controller.get_arm_qpos().copy())
            commanded_joints.append(q_command.copy())
            timestamps.append(control_tick / CTRL_FREQ_HZ)

        eef_waypoint, _ = controller.get_eef_pose()
        actual_at_waypoints.append(eef_waypoint.copy())
        joint_at_waypoints.append(controller.get_arm_qpos().copy())

        if verbose and (waypoint_index == 1 or waypoint_index % max(1, int(show_every)) == 0 or waypoint_index == len(reference) - 1):
            tracking_error = np.linalg.norm(eef_waypoint - target)
            print(
                f"{name:16s} waypoint={waypoint_index:03d}/{len(reference)-1:03d} | "
                f"track={1000.0 * tracking_error:7.2f} mm | "
                f"ik_pe={1000.0 * ik_position_error[-1]:7.2f} mm | "
                f"status={ik_status[-1]}"
            )

    wall_time = time.perf_counter() - wall_start
    log = {
        "name": name,
        "space": "cartesian",
        "metadata": metadata,
        "command_regime": COMMAND_REGIME,
        "orientation_policy": CARTESIAN_ORIENTATION_POLICY,
        "fixed_orientation_quat_wxyz": np.asarray(initial_quat, dtype=float).copy(),
        "state_logging_frequency_hz": float(STATE_LOGGING_FREQUENCY_HZ),
        "physics_substep_logging": False,
        "reference_waypoints": reference.copy(),
        "reference_eef": np.asarray(reference_dense, dtype=float),
        "actual_eef": np.asarray(actual_eef, dtype=float),
        "actual_eef_at_waypoints": np.asarray(actual_at_waypoints, dtype=float),
        "actual_joints": np.asarray(actual_joints, dtype=float),
        "commanded_joints": np.asarray(commanded_joints, dtype=float),
        "actual_joints_at_waypoints": np.asarray(joint_at_waypoints, dtype=float),
        "timestamps_s": np.asarray(timestamps, dtype=float),
        "ik_status": np.asarray(ik_status, dtype=str),
        "ik_position_error_m": np.asarray(ik_position_error, dtype=float),
        "ik_rotation_error_rad": np.asarray(ik_rotation_error, dtype=float),
        "ik_ok": np.asarray(ik_ok, dtype=bool),
        "ik_accepted": np.asarray(ik_accepted, dtype=bool),
        "ik_solve_time_ms": np.asarray(ik_solve_time_ms, dtype=float),
        "wall_time_s": float(wall_time),
        "simulated_time_s": float(timestamps[-1]),
        "controller_frequency_hz": float(CTRL_FREQ_HZ),
        "physics_frequency_hz": float(SIM_FREQ_HZ),
        "control_ticks": int(control_tick),
        "wall_control_rate_hz": float(control_tick / wall_time) if wall_time > 0.0 else np.nan,
        "ctrl_ticks_per_waypoint": int(ctrl_ticks_per_waypoint),
        "joint_command_clip_events": int(joint_command_clip_events),
    }
    if verbose:
        print(f"Finished {name}: {control_tick} control ticks, {wall_time:.2f} s wall time")
    return log


def execute_joint_trajectory(
    reference_trajectory,
    *,
    controller=None,
    ctrl_ticks_per_waypoint: int = CTRL_TICKS_PER_JOINT_WAYPOINT,
    verbose: bool = True,
    show_every: int = 15,
) -> dict:
    '''Execute a fixed 7-joint trajectory; IK fields are explicitly not applicable.'''
    controller = ctrl if controller is None else controller
    trajectory = reference_trajectory if isinstance(reference_trajectory, dict) else {}
    reference_q = np.asarray(trajectory.get("points", reference_trajectory), dtype=float)
    if reference_q.ndim != 2 or reference_q.shape[1] != 7 or len(reference_q) < 2:
        raise ValueError("reference_trajectory must contain at least two 7-joint points")
    if not np.all(np.isfinite(reference_q)):
        raise ValueError("reference_trajectory contains non-finite values")

    name = str(trajectory.get("name", "joint_trajectory"))
    metadata = dict(trajectory.get("metadata", {}))
    initial_eef, _ = controller.get_eef_pose()
    initial_q = controller.get_arm_qpos()

    reference_eef = [_forward_kinematics(controller, reference_q[0])]
    actual_eef = [initial_eef.copy()]
    reference_joints = [reference_q[0].copy()]
    actual_joints = [initial_q.copy()]
    commanded_joints = [initial_q.copy()]
    timestamps = [0.0]
    control_tick = 0
    joint_command_clip_events = int(metadata.get("generator_clip_events", 0))
    wall_start = time.perf_counter()

    for waypoint_index, q_goal in enumerate(reference_q[1:], start=1):
        q_start = controller.get_arm_qpos().copy()
        previous_q = reference_q[waypoint_index - 1]
        for local_tick in range(int(ctrl_ticks_per_waypoint)):
            alpha = _smoothstep((local_tick + 1) / int(ctrl_ticks_per_waypoint))
            q_reference_raw = (1.0 - alpha) * previous_q + alpha * q_goal
            q_command_raw = (1.0 - alpha) * q_start + alpha * q_goal
            q_reference_now = clamp_joints(q_reference_raw)
            q_command = clamp_joints(q_command_raw)
            joint_command_clip_events += int(np.any(np.abs(q_command - q_command_raw) > 1e-12))
            _run_control_tick(controller, q_command)
            control_tick += 1

            eef_now, _ = controller.get_eef_pose()
            reference_eef.append(_forward_kinematics(controller, q_reference_now))
            actual_eef.append(eef_now.copy())
            reference_joints.append(q_reference_now.copy())
            actual_joints.append(controller.get_arm_qpos().copy())
            commanded_joints.append(q_command.copy())
            timestamps.append(control_tick / CTRL_FREQ_HZ)

        if verbose and (waypoint_index == 1 or waypoint_index % max(1, int(show_every)) == 0 or waypoint_index == len(reference_q) - 1):
            joint_error = np.max(np.abs(controller.get_arm_qpos() - q_goal))
            print(
                f"{name:16s} waypoint={waypoint_index:03d}/{len(reference_q)-1:03d} | "
                f"max_joint_error={joint_error:7.4f} rad"
            )

    wall_time = time.perf_counter() - wall_start
    n_steps = len(reference_q) - 1
    return {
        "name": name,
        "space": "joint",
        "metadata": metadata,
        "command_regime": COMMAND_REGIME,
        "orientation_policy": "not_applicable_joint_space",
        "state_logging_frequency_hz": float(STATE_LOGGING_FREQUENCY_HZ),
        "physics_substep_logging": False,
        "reference_waypoints": reference_q.copy(),
        "reference_eef": np.asarray(reference_eef, dtype=float),
        "actual_eef": np.asarray(actual_eef, dtype=float),
        "actual_eef_at_waypoints": np.asarray(actual_eef, dtype=float),
        "reference_joints": np.asarray(reference_joints, dtype=float),
        "actual_joints": np.asarray(actual_joints, dtype=float),
        "commanded_joints": np.asarray(commanded_joints, dtype=float),
        "actual_joints_at_waypoints": np.asarray(actual_joints, dtype=float),
        "timestamps_s": np.asarray(timestamps, dtype=float),
        "ik_status": np.full(n_steps, "not_applicable_joint_space", dtype=str),
        "ik_position_error_m": np.full(n_steps, np.nan),
        "ik_rotation_error_rad": np.full(n_steps, np.nan),
        "ik_ok": np.full(n_steps, False, dtype=bool),
        "ik_accepted": np.full(n_steps, False, dtype=bool),
        "ik_solve_time_ms": np.full(n_steps, np.nan),
        "wall_time_s": float(wall_time),
        "simulated_time_s": float(timestamps[-1]),
        "controller_frequency_hz": float(CTRL_FREQ_HZ),
        "physics_frequency_hz": float(SIM_FREQ_HZ),
        "control_ticks": int(control_tick),
        "wall_control_rate_hz": float(control_tick / wall_time) if wall_time > 0.0 else np.nan,
        "ctrl_ticks_per_waypoint": int(ctrl_ticks_per_waypoint),
        "joint_command_clip_events": int(joint_command_clip_events),
    }


## 6. Motion metrics

In [ ]:
def path_length(points) -> float:
    points = np.asarray(points, dtype=float)
    if len(points) < 2:
        return 0.0
    return float(np.sum(np.linalg.norm(np.diff(points, axis=0), axis=1)))


def derivative_profile(values, dt: float) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    if len(values) < 2:
        return np.zeros_like(values)
    edge_order = 2 if len(values) >= 3 else 1
    return np.gradient(values, float(dt), axis=0, edge_order=edge_order)


def motion_profiles(points, dt: float) -> dict:
    position = np.asarray(points, dtype=float)
    velocity = derivative_profile(position, dt)
    acceleration = derivative_profile(velocity, dt)
    jerk = derivative_profile(acceleration, dt)
    return {
        "velocity": velocity,
        "speed": np.linalg.norm(velocity, axis=1),
        "acceleration": acceleration,
        "acceleration_magnitude": np.linalg.norm(acceleration, axis=1),
        "jerk": jerk,
        "jerk_magnitude": np.linalg.norm(jerk, axis=1),
    }


def compute_motion_metrics(log: dict) -> dict:
    reference = np.asarray(log["reference_eef"], dtype=float)
    actual = np.asarray(log["actual_eef"], dtype=float)
    joints = np.asarray(log["actual_joints"], dtype=float)
    if reference.shape != actual.shape:
        raise ValueError("Reference and actual EEF arrays must be synchronized")

    dt = 1.0 / float(log["controller_frequency_hz"])
    cartesian_error = np.linalg.norm(actual - reference, axis=1)
    reference_length = path_length(reference)
    actual_length = path_length(actual)
    actual_profile = motion_profiles(actual, dt)
    reference_profile = motion_profiles(reference, dt)
    joint_velocity = derivative_profile(joints, dt)
    joint_acceleration = derivative_profile(joint_velocity, dt)
    joint_steps = np.diff(joints, axis=0) if len(joints) >= 2 else np.zeros((0, joints.shape[1]))
    joint_clearance = np.minimum(joints - JOINT_LIMITS[:, 0], JOINT_LIMITS[:, 1] - joints)
    minimum_joint_margin_per_joint = np.min(joint_clearance, axis=0)

    duration = float(log["simulated_time_s"])
    integrated_squared_jerk = float(np.trapz(actual_profile["jerk_magnitude"] ** 2, dx=dt))
    normalized_jerk = (
        integrated_squared_jerk * duration**5 / actual_length**2
        if duration > 0.0 and actual_length > 0.0
        else np.nan
    )

    metrics = {
        "name": log["name"],
        "space": log["space"],
        "samples": int(len(actual)),
        "controller_frequency_hz": float(log["controller_frequency_hz"]),
        "simulated_time_s": duration,
        "wall_time_s": float(log["wall_time_s"]),
        "wall_control_rate_hz": float(log["wall_control_rate_hz"]),
        "mean_cartesian_error_mm": 1000.0 * float(np.mean(cartesian_error)),
        "rmse_cartesian_error_mm": 1000.0 * float(np.sqrt(np.mean(cartesian_error**2))),
        "max_cartesian_error_mm": 1000.0 * float(np.max(cartesian_error)),
        "reference_path_length_m": reference_length,
        "actual_path_length_m": actual_length,
        "path_efficiency": reference_length / actual_length if actual_length > 0.0 else np.nan,
        "path_length_ratio_actual_to_reference": actual_length / reference_length if reference_length > 0.0 else np.nan,
        "max_eef_speed_m_s": float(np.max(actual_profile["speed"])),
        "rms_eef_speed_m_s": float(np.sqrt(np.mean(actual_profile["speed"] ** 2))),
        "max_eef_acceleration_m_s2": float(np.max(actual_profile["acceleration_magnitude"])),
        "rms_eef_acceleration_m_s2": float(np.sqrt(np.mean(actual_profile["acceleration_magnitude"] ** 2))),
        "max_eef_jerk_m_s3": float(np.max(actual_profile["jerk_magnitude"])),
        "rms_eef_jerk_m_s3": float(np.sqrt(np.mean(actual_profile["jerk_magnitude"] ** 2))),
        "integrated_squared_jerk": integrated_squared_jerk,
        "normalized_jerk": float(normalized_jerk),
        "max_joint_velocity_rad_s": float(np.max(np.abs(joint_velocity))),
        "max_joint_velocity_per_joint_rad_s": np.max(np.abs(joint_velocity), axis=0),
        "max_joint_acceleration_rad_s2": float(np.max(np.abs(joint_acceleration))),
        "max_joint_acceleration_per_joint_rad_s2": np.max(np.abs(joint_acceleration), axis=0),
        "joint_discontinuity_rad": float(np.max(np.abs(joint_steps))) if joint_steps.size else 0.0,
        "minimum_joint_limit_margin_rad": float(np.min(minimum_joint_margin_per_joint)),
        "minimum_joint_limit_margin_per_joint_rad": minimum_joint_margin_per_joint,
        "joint_command_clip_events": int(log.get("joint_command_clip_events", 0)),
        "joint_command_clip_rate": float(log.get("joint_command_clip_events", 0) / max(int(log.get("control_ticks", 0)), 1)),
    }

    if log["space"] == "cartesian":
        ik_ok = np.asarray(log["ik_ok"], dtype=bool)
        ik_status = np.asarray(log["ik_status"], dtype=str)
        metrics.update({
            "solver_success_rate": float(np.mean(ik_ok)) if len(ik_ok) else np.nan,
            "ik_non_hold_rate": float(np.mean(ik_status != "hold_current_no_safe_candidate")) if len(ik_status) else np.nan,
            "ik_status_counts": {status: int(np.sum(ik_status == status)) for status in np.unique(ik_status)},
            "mean_ik_position_error_mm": 1000.0 * float(np.mean(log["ik_position_error_m"])),
            "max_ik_position_error_mm": 1000.0 * float(np.max(log["ik_position_error_m"])),
            "mean_ik_rotation_error_rad": float(np.mean(log["ik_rotation_error_rad"])),
            "max_ik_rotation_error_rad": float(np.max(log["ik_rotation_error_rad"])),
            "mean_ik_solve_time_ms": float(np.mean(log["ik_solve_time_ms"])),
            "p95_ik_solve_time_ms": float(np.percentile(log["ik_solve_time_ms"], 95)),
            "max_ik_solve_time_ms": float(np.max(log["ik_solve_time_ms"])),
        })
    else:
        reference_joints = np.asarray(log["reference_joints"], dtype=float)
        joint_error = joints - reference_joints
        metrics.update({
            "solver_success_rate": np.nan,
            "ik_non_hold_rate": np.nan,
            "ik_status_counts": {"not_applicable_joint_space": int(len(log["ik_status"]))},
            "mean_ik_position_error_mm": np.nan,
            "max_ik_position_error_mm": np.nan,
            "mean_ik_rotation_error_rad": np.nan,
            "max_ik_rotation_error_rad": np.nan,
            "mean_ik_solve_time_ms": np.nan,
            "p95_ik_solve_time_ms": np.nan,
            "max_ik_solve_time_ms": np.nan,
            "joint_tracking_rmse_rad": float(np.sqrt(np.mean(joint_error**2))),
            "max_joint_tracking_error_rad": float(np.max(np.abs(joint_error))),
        })

    log["profiles"] = {"actual": actual_profile, "reference": reference_profile}
    log["cartesian_error_m"] = cartesian_error
    log["joint_velocity_rad_s"] = joint_velocity
    log["joint_acceleration_rad_s2"] = joint_acceleration
    log["metrics"] = metrics
    return metrics


In [ ]:
def fit_circle_3d(points) -> dict:
    '''Offline least-squares 3-D circle fit used only for evaluation.'''
    points = np.asarray(points, dtype=float)
    if points.ndim != 2 or points.shape[1] != 3 or len(points) < 5:
        raise ValueError("Circle fit needs at least five XYZ points")
    center_of_points = np.mean(points, axis=0)
    centered = points - center_of_points
    _, _, vh = np.linalg.svd(centered, full_matrices=False)
    basis_u, basis_v, normal = vh
    uv = np.column_stack([centered @ basis_u, centered @ basis_v])
    x, y = uv[:, 0], uv[:, 1]
    system = np.column_stack([2.0 * x, 2.0 * y, np.ones_like(x)])
    rhs = x * x + y * y
    cx, cy, constant = np.linalg.lstsq(system, rhs, rcond=None)[0]
    radius = float(np.sqrt(max(constant + cx * cx + cy * cy, 0.0)))
    center_3d = center_of_points + cx * basis_u + cy * basis_v
    radial_distance = np.sqrt((x - cx) ** 2 + (y - cy) ** 2)
    radial_residual = radial_distance - radius
    plane_residual = (points - center_3d) @ normal
    return {
        "center_3d": center_3d,
        "basis_u": basis_u,
        "basis_v": basis_v,
        "normal": normal,
        "radius_m": radius,
        "radial_residual_m": radial_residual,
        "radial_rmse_m": float(np.sqrt(np.mean(radial_residual**2))),
        "planarity_rmse_m": float(np.sqrt(np.mean(plane_residual**2))),
    }


def compute_circle_metrics(log: dict) -> dict:
    reference = np.asarray(log["reference_eef"], dtype=float)
    actual = np.asarray(log["actual_eef"], dtype=float)
    reference_fit = fit_circle_3d(reference)
    actual_fit = fit_circle_3d(actual)

    offset = actual - reference_fit["center_3d"]
    u = offset @ reference_fit["basis_u"]
    v = offset @ reference_fit["basis_v"]
    radial_tracking_error = np.sqrt(u * u + v * v) - reference_fit["radius_m"]
    plane_tracking_error = offset @ reference_fit["normal"]

    circle_metrics = {
        "reference_radius_m": reference_fit["radius_m"],
        "fitted_actual_radius_m": actual_fit["radius_m"],
        "radius_error_mm": 1000.0 * (actual_fit["radius_m"] - reference_fit["radius_m"]),
        "radial_tracking_rmse_mm": 1000.0 * float(np.sqrt(np.mean(radial_tracking_error**2))),
        "radial_tracking_max_mm": 1000.0 * float(np.max(np.abs(radial_tracking_error))),
        "fitted_circle_radial_rmse_mm": 1000.0 * actual_fit["radial_rmse_m"],
        "planarity_rmse_mm": 1000.0 * actual_fit["planarity_rmse_m"],
        "reference_plane_tracking_rmse_mm": 1000.0 * float(np.sqrt(np.mean(plane_tracking_error**2))),
        "closure_error_mm": 1000.0 * float(np.linalg.norm(actual[-1] - actual[0])),
    }
    log["circle_fit"] = actual_fit
    log["circle_reference_fit"] = reference_fit
    log["radial_tracking_error_m"] = radial_tracking_error
    log["circle_metrics"] = circle_metrics
    if "metrics" in log:
        log["metrics"].update(circle_metrics)
    return circle_metrics


def repeatability_metrics(logs) -> dict:
    if len(logs) < 2:
        raise ValueError("Repeatability requires at least two trials")
    trajectories = [np.asarray(log["actual_eef"], dtype=float) for log in logs]
    if len({trajectory.shape for trajectory in trajectories}) != 1:
        raise ValueError("Repeatability trajectories must have identical synchronized shapes")
    trajectory_stack = np.stack(trajectories, axis=0)
    mean_trajectory = np.mean(trajectory_stack, axis=0)
    trajectory_deviation = np.linalg.norm(trajectory_stack - mean_trajectory[None, :, :], axis=2)

    final_positions = trajectory_stack[:, -1, :]
    final_targets = np.vstack([np.asarray(log["reference_eef"][-1], dtype=float) for log in logs])
    centroid = np.mean(final_positions, axis=0)
    deviations = np.linalg.norm(final_positions - centroid, axis=1)
    final_target_errors = np.linalg.norm(final_positions - final_targets, axis=1)
    pairwise = np.array([
        np.linalg.norm(final_positions[i] - final_positions[j])
        for i in range(len(final_positions))
        for j in range(i + 1, len(final_positions))
    ])
    return {
        "n_trials": len(logs),
        "final_positions_m": final_positions,
        "mean_final_position_m": centroid,
        "mean_trajectory_m": mean_trajectory,
        "trajectory_repeatability_rmse_mm": 1000.0 * float(np.sqrt(np.mean(trajectory_deviation**2))),
        "trajectory_repeatability_max_mm": 1000.0 * float(np.max(trajectory_deviation)),
        "final_deviation_mean_mm": 1000.0 * float(np.mean(deviations)),
        "final_deviation_std_mm": 1000.0 * float(np.std(deviations, ddof=1)),
        "final_deviation_max_mm": 1000.0 * float(np.max(deviations)),
        "pairwise_final_distance_mean_mm": 1000.0 * float(np.mean(pairwise)),
        "pairwise_final_distance_max_mm": 1000.0 * float(np.max(pairwise)),
        "final_target_error_mean_mm": 1000.0 * float(np.mean(final_target_errors)),
        "final_target_error_std_mm": 1000.0 * float(np.std(final_target_errors, ddof=1)),
    }


## 7. Visualization

In [ ]:
def selected_figure_style() -> dict:
    if FIGURE_STYLE_MODE not in FIGURE_STYLE_PRESETS:
        choices = ", ".join(FIGURE_STYLE_PRESETS)
        raise ValueError(f"Unknown FIGURE_STYLE_MODE={FIGURE_STYLE_MODE!r}; choose from: {choices}")
    return FIGURE_STYLE_PRESETS[FIGURE_STYLE_MODE]


def style_axis(ax, *, legend: bool = False) -> None:
    style = selected_figure_style()
    ax.grid(True, which="major", color=".8", linewidth=0.7, alpha=style["grid_alpha"])
    ax.tick_params(
        axis="both", which="major", labelsize=8.5,
        width=style["tick_width"], length=style["tick_length"],
    )
    ax.set_title(ax.get_title(), fontsize=style["title_size"],
                 fontweight="semibold", pad=6)
    ax.xaxis.label.set_size(10)
    ax.yaxis.label.set_size(10)
    if hasattr(ax, "zaxis"):
        ax.zaxis.label.set_size(10)
    else:
        ax.yaxis.set_major_locator(MaxNLocator(5))
    for spine in getattr(ax, "spines", {}).values():
        spine.set_color("0.15")
        spine.set_linewidth(0.8)
    if legend:
        handles, labels = ax.get_legend_handles_labels()
        if handles:
            loc = "upper left" if hasattr(ax, "zaxis") else "best"
            ax.legend(handles, labels, fontsize=style["legend_size"], loc=loc)


def thesis_title(fig, main: str, *, top: float = 0.94, y: float = 0.975,
                 layout: bool = True) -> None:
    """Apply the same bold figure-title treatment as the evaluation notebook."""
    fig.suptitle(main, fontsize=14, fontweight="bold", y=y, va="top")
    # The explicit `rect` below already reserves title space. Excluding the
    # suptitle from tight-layout prevents Matplotlib from reserving it twice.
    fig._suptitle.set_in_layout(False)
    if layout:
        fig.tight_layout(rect=(0, 0, 1, top))
    # Include it again when savefig computes the tight export bounding box.
    fig._suptitle.set_in_layout(True)


def set_3d_axes_equal(ax, points) -> None:
    points = np.asarray(points, dtype=float).reshape(-1, 3)
    lower = np.min(points, axis=0)
    upper = np.max(points, axis=0)
    center = 0.5 * (lower + upper)
    radius = 0.5 * max(float(np.max(upper - lower)), 1e-6) * 1.15
    ax.set_xlim(center[0] - radius, center[0] + radius)
    ax.set_ylim(center[1] - radius, center[1] + radius)
    ax.set_zlim(center[2] - radius, center[2] + radius)
    if hasattr(ax, "set_box_aspect"):
        ax.set_box_aspect((1, 1, 1))


def _draw_eef_trajectory(ax, log: dict) -> None:
    reference = log["reference_eef"]
    actual = log["actual_eef"]
    style = selected_figure_style()
    ax.plot(reference[:, 0], reference[:, 1], reference[:, 2], "--", color=C_RED,
            linewidth=style["reference_line_width"], label="Reference")
    ax.plot(actual[:, 0], actual[:, 1], actual[:, 2], color=C_BLUE,
            linewidth=style["line_width"], label="Actual")
    ax.scatter(*actual[0], s=50, color=C_GREEN, edgecolors="white", linewidth=0.8,
               depthshade=False, zorder=5, label="Start")
    ax.scatter(*actual[-1], s=50, marker="X", color="#C92A2A", edgecolors="white",
               linewidth=0.8, depthshade=False, zorder=5, label="End")
    ax.set(xlabel="x [m]", ylabel="y [m]", zlabel="z [m]", title="EEF trajectory")
    ax.zaxis.labelpad = 2
    set_3d_axes_equal(ax, np.vstack([reference, actual]))
    ax.view_init(elev=22, azim=-58)
    style_axis(ax, legend=True)


def _draw_position_error(ax, log: dict) -> None:
    style = selected_figure_style()
    ax.plot(log["timestamps_s"], 1000.0 * log["cartesian_error_m"], color=C_BLUE,
            linewidth=style["line_width"], label="Tracking error")
    ax.set(xlabel="Time [s]", ylabel="Position error [mm]", title="Cartesian tracking error")
    style_axis(ax, legend=True)


def _draw_joint_trajectories(ax, log: dict) -> None:
    time_axis = log["timestamps_s"]
    joint_colors = (C_BLUE, C_RED, C_GREEN, C_PURPLE, C_ORANGE, C_CYAN, C_GREY)
    for joint_index, joint_name in enumerate(ARM_JOINT_NAMES_BARE):
        ax.plot(time_axis, log["actual_joints"][:, joint_index],
                color=joint_colors[joint_index % len(joint_colors)],
                linewidth=selected_figure_style()["line_width"],
                label=joint_name.replace("_joint", ""))
    if "reference_joints" in log:
        for joint_index in range(log["reference_joints"].shape[1]):
            ax.plot(time_axis, log["reference_joints"][:, joint_index], "--", color="0.15",
                    linewidth=selected_figure_style()["reference_line_width"], alpha=0.55,
                    label="Reference" if joint_index == 0 else None)
    ax.set(xlabel="Time [s]", ylabel="Joint position [rad]", title="Measured joint trajectories")
    style_axis(ax, legend=True)
    ax.legend(fontsize=selected_figure_style()["legend_size"], ncol=2)


def _draw_velocity(ax, log: dict) -> None:
    time_axis = log["timestamps_s"]
    profiles = log["profiles"]
    style = selected_figure_style()
    ax.plot(time_axis, profiles["reference"]["speed"], "--", color=C_RED,
            linewidth=style["reference_line_width"], label="Reference")
    ax.plot(time_axis, profiles["actual"]["speed"], color=C_BLUE,
            linewidth=style["line_width"], label="Actual")
    ax.set(xlabel="Time [s]", ylabel="Speed [m/s]", title="EEF velocity profile")
    style_axis(ax, legend=True)


def _draw_acceleration(ax, log: dict) -> None:
    time_axis = log["timestamps_s"]
    profiles = log["profiles"]
    style = selected_figure_style()
    ax.plot(time_axis, profiles["reference"]["acceleration_magnitude"], "--", color=C_RED,
            linewidth=style["reference_line_width"], label="Reference")
    ax.plot(time_axis, profiles["actual"]["acceleration_magnitude"], color=C_BLUE,
            linewidth=style["line_width"], label="Actual")
    ax.set(xlabel="Time [s]", ylabel="Acceleration [m/s²]", title="EEF acceleration profile")
    style_axis(ax, legend=True)


def _draw_jerk(ax, log: dict) -> None:
    time_axis = log["timestamps_s"]
    profiles = log["profiles"]
    style = selected_figure_style()
    ax.plot(time_axis, profiles["reference"]["jerk_magnitude"], "--", color=C_RED,
            linewidth=style["reference_line_width"], label="Reference")
    ax.plot(time_axis, profiles["actual"]["jerk_magnitude"], color=C_BLUE,
            linewidth=style["line_width"], label="Actual")
    ax.set(xlabel="Time [s]", ylabel="Jerk [m/s³]", title="EEF jerk profile")
    style_axis(ax, legend=True)


def make_eef_trajectory_figure(log: dict):
    fig = plt.figure(figsize=selected_figure_style()["trajectory_size"])
    ax = fig.add_subplot(111, projection="3d")
    _draw_eef_trajectory(ax, log)
    ax.set_title("")
    thesis_title(fig, "EEF trajectory", layout=False)
    fig.subplots_adjust(left=0.02, right=0.90, bottom=0.03, top=0.94)
    return fig


def _make_single_panel_figure(log: dict, draw_function, figsize=None):
    if figsize is None:
        figsize = selected_figure_style()["single_size"]
    fig, ax = plt.subplots(figsize=figsize)
    draw_function(ax, log)
    title = ax.get_title()
    ax.set_title("")
    thesis_title(fig, title, top=0.90)
    return fig


def make_position_error_figure(log: dict):
    return _make_single_panel_figure(log, _draw_position_error)


def make_joint_trajectories_figure(log: dict):
    return _make_single_panel_figure(log, _draw_joint_trajectories, figsize=tuple(value * 1.15 for value in selected_figure_style()["single_size"]))


def make_velocity_figure(log: dict):
    return _make_single_panel_figure(log, _draw_velocity)


def make_acceleration_figure(log: dict):
    return _make_single_panel_figure(log, _draw_acceleration)


def make_jerk_figure(log: dict):
    return _make_single_panel_figure(log, _draw_jerk)


def make_benchmark_dashboard(log: dict):
    if "metrics" not in log:
        compute_motion_metrics(log)
    fig = plt.figure(figsize=selected_figure_style()["dashboard_size"])
    _draw_eef_trajectory(fig.add_subplot(2, 3, 1, projection="3d"), log)
    _draw_position_error(fig.add_subplot(2, 3, 2), log)
    _draw_joint_trajectories(fig.add_subplot(2, 3, 3), log)
    _draw_velocity(fig.add_subplot(2, 3, 4), log)
    _draw_acceleration(fig.add_subplot(2, 3, 5), log)
    _draw_jerk(fig.add_subplot(2, 3, 6), log)
    thesis_title(fig, log["name"].replace("_", " ").title(), top=0.93)
    return fig


def plot_benchmark(log: dict) -> None:
    display(make_benchmark_dashboard(log))
    plt.close()


def _fitted_circle_points(log: dict) -> np.ndarray:
    if "circle_fit" not in log:
        compute_circle_metrics(log)
    fit = log["circle_fit"]
    theta = np.linspace(0.0, 2.0 * np.pi, 300)
    return (
        fit["center_3d"]
        + fit["radius_m"] * np.cos(theta)[:, None] * fit["basis_u"]
        + fit["radius_m"] * np.sin(theta)[:, None] * fit["basis_v"]
    )


def _draw_circle_fit(ax, log: dict) -> None:
    reference = log["reference_eef"]
    actual = log["actual_eef"]
    fitted_circle = _fitted_circle_points(log)
    style = selected_figure_style()
    ax.plot(reference[:, 0], reference[:, 1], reference[:, 2], "--", color=C_RED,
            linewidth=style["reference_line_width"], label="Reference")
    ax.plot(actual[:, 0], actual[:, 1], actual[:, 2], color=C_BLUE,
            linewidth=style["line_width"], label="Actual")
    ax.plot(fitted_circle[:, 0], fitted_circle[:, 1], fitted_circle[:, 2], ":", color=C_GREEN,
            linewidth=style["line_width"], label="Offline best fit")
    ax.set(xlabel="x [m]", ylabel="y [m]", zlabel="z [m]",
           title="Circle trajectory and best fit")
    ax.zaxis.labelpad = 2
    set_3d_axes_equal(ax, np.vstack([reference, actual, fitted_circle]))
    ax.view_init(elev=22, azim=-58)
    style_axis(ax, legend=True)


def _draw_circle_radial_error(ax, log: dict) -> None:
    ax.plot(log["timestamps_s"], 1000.0 * log["radial_tracking_error_m"], color=C_BLUE,
            linewidth=selected_figure_style()["line_width"], label="Radial error")
    ax.axhline(0.0, color="0.15", linewidth=1)
    ax.set(xlabel="Time [s]", ylabel="Radial tracking error [mm]", title="Radial error relative to reference circle")
    style_axis(ax, legend=True)


def make_circle_fit_figure(log: dict):
    fig = plt.figure(figsize=selected_figure_style()["trajectory_size"])
    ax = fig.add_subplot(111, projection="3d")
    _draw_circle_fit(ax, log)
    ax.set_title("")
    thesis_title(fig, "Circle trajectory and best fit", layout=False)
    fig.subplots_adjust(left=0.02, right=0.90, bottom=0.03, top=0.94)
    return fig


def make_circle_radial_error_figure(log: dict):
    return _make_single_panel_figure(log, _draw_circle_radial_error)


def make_circle_diagnostics_dashboard(log: dict):
    fig = plt.figure(figsize=tuple(value * 0.9 for value in selected_figure_style()["dashboard_size"]))
    _draw_circle_fit(fig.add_subplot(1, 2, 1, projection="3d"), log)
    _draw_circle_radial_error(fig.add_subplot(1, 2, 2), log)
    fig.tight_layout()
    return fig


def plot_circle_diagnostics(log: dict) -> None:
    display(make_circle_diagnostics_dashboard(log))
    plt.close()


def _draw_repeatability_endpoints(ax, result: dict) -> None:
    final_positions = result["metrics"]["final_positions_m"]
    center = result["metrics"]["mean_final_position_m"]
    ax.scatter(final_positions[:, 0], final_positions[:, 1], final_positions[:, 2],
               color=C_BLUE, edgecolors="white", linewidth=0.8, s=55, label="Trial endpoints")
    ax.scatter(*center, marker="x", color=C_RED, linewidth=1.8, s=90, label="Mean endpoint")
    ax.set(xlabel="x [m]", ylabel="y [m]", zlabel="z [m]", title="Repeatability endpoints")
    ax.zaxis.labelpad = 2
    set_3d_axes_equal(ax, np.vstack([final_positions, center[None, :]]))
    ax.view_init(elev=22, azim=-58)
    style_axis(ax, legend=True)


def _draw_repeatability_trajectories(ax, result: dict) -> None:
    reference = result["logs"][0]["reference_eef"]
    for index, log in enumerate(result["logs"]):
        actual = log["actual_eef"]
        ax.plot(actual[:, 0], actual[:, 1], actual[:, 2], color=C_BLUE,
                linewidth=selected_figure_style()["line_width"], alpha=0.35,
                label=f"Trial {index + 1}")
    mean_trajectory = result["metrics"]["mean_trajectory_m"]
    ax.plot(reference[:, 0], reference[:, 1], reference[:, 2], "--", color=C_RED,
            linewidth=selected_figure_style()["reference_line_width"], label="Reference")
    ax.plot(mean_trajectory[:, 0], mean_trajectory[:, 1], mean_trajectory[:, 2], color=C_GREEN,
            linewidth=selected_figure_style()["line_width"] + 0.8, label="Trial mean")
    all_points = np.vstack([reference, mean_trajectory] + [log["actual_eef"] for log in result["logs"]])
    ax.set(xlabel="x [m]", ylabel="y [m]", zlabel="z [m]", title="Whole-trajectory repeatability")
    ax.zaxis.labelpad = 2
    set_3d_axes_equal(ax, all_points)
    ax.view_init(elev=22, azim=-58)
    style_axis(ax, legend=True)


def make_repeatability_endpoints_figure(result: dict):
    fig = plt.figure(figsize=selected_figure_style()["trajectory_size"])
    ax = fig.add_subplot(111, projection="3d")
    _draw_repeatability_endpoints(ax, result)
    ax.set_title("")
    thesis_title(fig, "Repeatability endpoints", layout=False)
    fig.subplots_adjust(left=0.02, right=0.90, bottom=0.03, top=0.94)
    return fig


def make_repeatability_trajectories_figure(result: dict):
    fig = plt.figure(figsize=selected_figure_style()["trajectory_size"])
    ax = fig.add_subplot(111, projection="3d")
    _draw_repeatability_trajectories(ax, result)
    ax.set_title("")
    thesis_title(fig, "Whole-trajectory repeatability", layout=False)
    fig.subplots_adjust(left=0.02, right=0.90, bottom=0.03, top=0.94)
    return fig


def make_repeatability_dashboard(result: dict):
    fig = plt.figure(figsize=tuple(value * 0.95 for value in selected_figure_style()["dashboard_size"]))
    _draw_repeatability_trajectories(fig.add_subplot(1, 2, 1, projection="3d"), result)
    _draw_repeatability_endpoints(fig.add_subplot(1, 2, 2, projection="3d"), result)
    thesis_title(fig, f"{result['reference']['name']} repeatability", top=0.93)
    return fig


def plot_repeatability(result: dict) -> None:
    display(make_repeatability_dashboard(result))
    plt.close()


def make_summary_comparison_figure(results: dict):
    names = list(results)
    rmse = [results[name]["metrics"]["rmse_cartesian_error_mm"] for name in names]
    efficiency = [results[name]["metrics"]["path_efficiency"] for name in names]
    cartesian_names = [name for name in names if results[name]["space"] == "cartesian"]
    strict_success = [100.0 * results[name]["metrics"]["solver_success_rate"] for name in cartesian_names]
    dashboard_width, dashboard_height = selected_figure_style()["dashboard_size"]
    fig, axes = plt.subplots(1, 3, figsize=(dashboard_width, 0.55 * dashboard_height))
    axes[0].bar(names, rmse, color=C_BLUE)
    axes[0].set(ylabel="RMSE [mm]", title="Cartesian tracking error")
    axes[1].bar(names, efficiency, color=C_GREEN)
    axes[1].axhline(1.0, color="0.15", linewidth=1, linestyle="--")
    axes[1].set(ylabel="Reference / actual path length", title="Path efficiency")
    axes[2].bar(cartesian_names, strict_success, color=C_PURPLE)
    axes[2].set(ylabel="Strict IK success [%]", title="Full-pose IK reliability", ylim=(0, 105))
    for axis in axes:
        style_axis(axis, legend=False)
        axis.tick_params(axis="x", rotation=45)
    thesis_title(fig, "SURENA motion benchmark comparison", top=0.90)
    return fig


def plot_summary_comparison(results: dict) -> None:
    display(make_summary_comparison_figure(results))
    plt.close()


In [ ]:
def _json_value(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.floating, np.integer)):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(key): _json_value(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [_json_value(item) for item in value]
    return value


def benchmark_mode_directory(mode_name: str, output_root: Path = BENCHMARK_OUTPUT_ROOT) -> Path:
    directory = Path(output_root) / str(mode_name)
    directory.mkdir(parents=True, exist_ok=True)
    return directory


def save_figure(fig, path: Path, *, dpi: int | None = None) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if dpi is None:
        dpi = selected_figure_style()["dpi"]
    fig.savefig(path, dpi=int(dpi), bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return path


def save_benchmark_figures(log: dict, directory: Path | None = None) -> dict:
    directory = benchmark_mode_directory(log["name"]) if directory is None else Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    builders = {
        "01_eef_trajectory_3d.png": make_eef_trajectory_figure,
        "02_position_error.png": make_position_error_figure,
        "03_joint_trajectories.png": make_joint_trajectories_figure,
        "04_velocity_profile.png": make_velocity_figure,
        "05_acceleration_profile.png": make_acceleration_figure,
        "06_jerk_profile.png": make_jerk_figure,
        "dashboard.png": make_benchmark_dashboard,
    }
    paths = {filename: save_figure(builder(log), directory / filename) for filename, builder in builders.items()}
    if log["name"] == "circle":
        circle_builders = {
            "07_circle_fit_3d.png": make_circle_fit_figure,
            "08_circle_radial_error.png": make_circle_radial_error_figure,
            "circle_diagnostics_dashboard.png": make_circle_diagnostics_dashboard,
        }
        paths.update({filename: save_figure(builder(log), directory / filename) for filename, builder in circle_builders.items()})
    return paths


def save_repeatability_figures(result: dict, directory: Path) -> dict:
    directory = Path(directory)
    builders = {
        "repeatability_trajectories_3d.png": make_repeatability_trajectories_figure,
        "repeatability_endpoints_3d.png": make_repeatability_endpoints_figure,
        "repeatability_dashboard.png": make_repeatability_dashboard,
    }
    return {filename: save_figure(builder(result), directory / filename) for filename, builder in builders.items()}


def save_benchmark_log(
    log: dict,
    output_root: Path = BENCHMARK_OUTPUT_ROOT,
    *,
    directory: Path | None = None,
    stem: str | None = None,
):
    directory = benchmark_mode_directory(log["name"], output_root) if directory is None else Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    stem = str(log["name"] if stem is None else stem)
    npz_path = directory / f"{stem}.npz"
    json_path = directory / f"{stem}.json"
    array_fields = {
        key: value for key, value in log.items()
        if isinstance(value, np.ndarray) and value.dtype.kind not in {"U", "S", "O"}
    }
    np.savez_compressed(npz_path, **array_fields)
    metadata = {
        "name": log["name"], "space": log["space"], "metadata": log["metadata"],
        "metrics": log.get("metrics", {}), "circle_metrics": log.get("circle_metrics", {}),
        "ik_status": log["ik_status"], "controller_config": CONTROLLER_CONFIG,
        "benchmark_geometry": BENCHMARK_GEOMETRY,
        "controller_frequency_hz": log["controller_frequency_hz"],
        "physics_frequency_hz": log["physics_frequency_hz"],
        "state_logging_frequency_hz": log.get("state_logging_frequency_hz"),
        "physics_substep_logging": log.get("physics_substep_logging", False),
        "command_regime": log.get("command_regime"), "orientation_policy": log.get("orientation_policy"),
        "fixed_orientation_quat_wxyz": log.get("fixed_orientation_quat_wxyz"),
        "joint_command_clip_events": log.get("joint_command_clip_events", 0),
    }
    json_path.write_text(json.dumps(_json_value(metadata), indent=2))
    return npz_path, json_path


## 8. Run the benchmark suite


In [ ]:
def build_cartesian_benchmark(name: str, start: np.ndarray) -> dict:
    geometry = BENCHMARK_GEOMETRY
    if name == "straight_line":
        return generate_line(start, delta=geometry["straight_delta_m"], n_points=geometry["line_points"], name=name)
    if name == "diagonal_line":
        return generate_line(start, delta=geometry["diagonal_delta_m"], n_points=geometry["line_points"], name=name)
    if name == "circle":
        return generate_circle(start, radius=geometry["circle_radius_m"], n_points=geometry["circle_points"])
    if name in ("square", "small_square"):
        return generate_square(
            start,
            side=(geometry["square_side_m"] if name == "square" else geometry["small_square_side_m"]),
            points_per_side=(geometry["square_points_per_side"] if name == "square" else geometry["small_square_points_per_side"]),
            name=name,
        )
    raise KeyError(name)


def build_joint_benchmarks() -> list[dict]:
    geometry = BENCHMARK_GEOMETRY
    common = {"home": HOME_QPOS, "points_per_segment": geometry["joint_points_per_segment"]}
    return [
        generate_joint_sweep(
            **common,
            joint_name="r_arm_pitch_joint",
            first_angle=geometry["shoulder_pitch_first_rad"],
            second_angle=geometry["shoulder_pitch_second_rad"],
            name="shoulder_pitch_joint_motion",
        ),
        generate_joint_sweep(
            **common,
            joint_name="r_elbow_pitch_joint",
            first_angle=geometry["elbow_flex_rad"],
            second_angle=geometry["elbow_extend_rad"],
            name="elbow_joint_motion",
        ),
        generate_joint_sweep(
            **common,
            joint_name="r_hand_pitch_joint",
            first_angle=geometry["wrist_pitch_first_rad"],
            second_angle=geometry["wrist_pitch_second_rad"],
            name="wrist_pitch_joint_motion",
        ),
    ]


def run_benchmark_suite(*, verbose: bool = True):
    global ctrl
    results = {}
    for name in ("straight_line", "diagonal_line", "circle", "square", "small_square"):
        print(f"\n=== {name} ===")
        ctrl = reset_robot(verbose=False)
        start, _ = ctrl.get_eef_pose()
        trajectory = build_cartesian_benchmark(name, start)
        log = execute_cartesian_trajectory(trajectory, controller=ctrl, verbose=verbose)
        compute_motion_metrics(log)
        if name == "circle":
            compute_circle_metrics(log)
        results[name] = log

    for joint_trajectory in build_joint_benchmarks():
        name = joint_trajectory["name"]
        print(f"\n=== {name} ===")
        ctrl = reset_robot(verbose=False)
        log = execute_joint_trajectory(joint_trajectory, controller=ctrl, verbose=verbose)
        compute_motion_metrics(log)
        results[name] = log

    return results, [log["metrics"] for log in results.values()]


RESULTS, SUMMARY = run_benchmark_suite(verbose=True)

summary_columns = [
    "name", "rmse_cartesian_error_mm", "max_cartesian_error_mm", "path_efficiency",
    "solver_success_rate", "joint_command_clip_events", "minimum_joint_limit_margin_rad",
    "max_joint_velocity_rad_s", "max_joint_acceleration_rad_s2", "wall_time_s",
]
display([{key: row.get(key, np.nan) for key in summary_columns} for row in SUMMARY])


In [ ]:
def resolve_display_benchmark_modes(results: dict, selection) -> list:
    available = list(results)
    if selection == "all":
        return available
    requested = [selection] if isinstance(selection, str) else list(selection)
    unknown = [name for name in requested if name not in results]
    if unknown:
        raise ValueError(f"Unknown DISPLAY_BENCHMARK_MODES: {unknown}. Available: {available}")
    return requested


SELECTED_DISPLAY_MODES = resolve_display_benchmark_modes(RESULTS, DISPLAY_BENCHMARK_MODES)
print("Figure style:", FIGURE_STYLE_MODE)
print("Displayed benchmark modes:", SELECTED_DISPLAY_MODES)

# Display the same dashboards that will later be exported as PNG files.
for mode_name in SELECTED_DISPLAY_MODES:
    plot_benchmark(RESULTS[mode_name])

if "circle" in SELECTED_DISPLAY_MODES:
    plot_circle_diagnostics(RESULTS["circle"])
plot_summary_comparison(RESULTS)


In [ ]:
def run_repeatability_test(
    reference_trajectory: dict,
    *,
    n_trials: int = REPEATABILITY_TRIALS,
    verbose: bool = True,
) -> dict:
    global ctrl
    benchmark_name = reference_trajectory["name"]
    logs = []
    for trial_index in range(int(n_trials)):
        print(f"\n{benchmark_name} repeatability trial {trial_index + 1}/{n_trials}")
        ctrl = reset_robot(verbose=False)
        trial_log = execute_cartesian_trajectory(
            reference_trajectory,
            controller=ctrl,
            verbose=verbose,
            show_every=len(reference_trajectory["points"]),
        )
        trial_log["name"] = f"repeatability_{benchmark_name}_trial_{trial_index + 1:02d}"
        compute_motion_metrics(trial_log)
        if benchmark_name == "circle":
            compute_circle_metrics(trial_log)
        logs.append(trial_log)
    return {"reference": reference_trajectory, "logs": logs, "metrics": repeatability_metrics(logs)}


# Each geometry reuses one identical absolute reference array across all trials.
REPEATABILITY_RESULTS = {}
for repeatability_name in REPEATABILITY_GEOMETRIES:
    ctrl = reset_robot(verbose=False)
    repeatability_start, _ = ctrl.get_eef_pose()
    repeatability_reference = build_cartesian_benchmark(repeatability_name, repeatability_start)
    result = run_repeatability_test(repeatability_reference)
    REPEATABILITY_RESULTS[repeatability_name] = result
    if repeatability_name in SELECTED_DISPLAY_MODES:
        scalar_metrics = {key: value for key, value in result["metrics"].items() if np.isscalar(value)}
        display({"trajectory": repeatability_name, **scalar_metrics})
        plot_repeatability(result)

# Compatibility alias for earlier analysis cells.
REPEATABILITY = REPEATABILITY_RESULTS["straight_line"]


In [ ]:
# Export one self-contained artifact directory per benchmark mode.
ARTIFACT_PATHS = {}
for mode_name, benchmark_log in RESULTS.items():
    mode_directory = benchmark_mode_directory(mode_name)
    data_paths = save_benchmark_log(benchmark_log, directory=mode_directory, stem=mode_name)
    figure_paths = save_benchmark_figures(benchmark_log, directory=mode_directory)
    ARTIFACT_PATHS[mode_name] = {"data": data_paths, "figures": figure_paths}

# Repeatability artifacts remain associated with their underlying motion mode.
for mode_name, result in REPEATABILITY_RESULTS.items():
    repeatability_directory = benchmark_mode_directory(mode_name) / "repeatability"
    repeatability_directory.mkdir(parents=True, exist_ok=True)
    trial_paths = []
    for trial_index, trial_log in enumerate(result["logs"], start=1):
        trial_paths.append(save_benchmark_log(
            trial_log,
            directory=repeatability_directory,
            stem=f"trial_{trial_index:02d}",
        ))
    repeatability_summary_path = repeatability_directory / "summary.json"
    repeatability_summary_path.write_text(json.dumps(_json_value(result["metrics"]), indent=2))
    repeatability_figure_paths = save_repeatability_figures(result, repeatability_directory)
    ARTIFACT_PATHS[mode_name]["repeatability"] = {
        "trials": trial_paths,
        "summary": repeatability_summary_path,
        "figures": repeatability_figure_paths,
    }

summary_directory = BENCHMARK_OUTPUT_ROOT / "_summary"
summary_directory.mkdir(parents=True, exist_ok=True)
summary_figure_path = save_figure(
    make_summary_comparison_figure(RESULTS),
    summary_directory / "benchmark_comparison_dashboard.png",
)

artifact_index_path = BENCHMARK_OUTPUT_ROOT / "artifact_index.json"
artifact_index_path.write_text(json.dumps(_json_value({
    "modes": ARTIFACT_PATHS,
    "summary_figure": summary_figure_path,
}), indent=2))

print("Exported benchmark artifact folders:")
for mode_name in RESULTS:
    print(" ", benchmark_mode_directory(mode_name))
print(" ", summary_directory)
print("Index:", artifact_index_path)


## Interpretation checklist

- High IK error or hold/fallback rates indicate kinematic feasibility or solver limitations.
- Low IK error but high Cartesian tracking error indicates dynamic joint-controller tracking limitations.
- Mean, p95, and maximum IK time distinguish typical from worst-case computational cost.
- Joint-command clipping events must remain zero for an unclipped benchmark; minimum joint-limit margin shows proximity even without clipping.
- Circle radial, plane, and closure errors quantify different geometric properties.
- Repeatability now includes whole-trajectory dispersion as well as endpoint spread.
- Zero spread under identical deterministic resets demonstrates numerical reproducibility, not robustness.
- The 20 Hz step-input regime is a stress condition and must not be presented as equivalent to a multi-tick downstream ramp.
- Controller-boundary logging cannot detect transients inside the 25 physics substeps between measurements.

Do not change geometry, solver tolerances, gains, sampling rates, or joint sweep amplitudes after inspecting scores without declaring a new experimental condition and rerunning the complete suite.


## 9. Thesis-ready quantitative tables


In [ ]:
import csv
import hashlib
import html as html_lib

from IPython.display import HTML, display


def _table_value(value, digits: int) -> str:
    if isinstance(value, dict):
        return html_lib.escape(", ".join(f"{key}: {item}" for key, item in value.items()))
    try:
        numeric = float(value)
    except (TypeError, ValueError):
        return html_lib.escape(str(value))
    if not np.isfinite(numeric):
        return "—"
    return f"{numeric:.{int(digits)}f}"


def display_metric_table(title: str, rows: list, columns: list) -> None:
    header = "".join(f"<th>{html_lib.escape(label)}</th>" for _, label, _ in columns)
    body = []
    for row in rows:
        cells = "".join(f"<td>{_table_value(row.get(key, np.nan), digits)}</td>" for key, _, digits in columns)
        body.append(f"<tr>{cells}</tr>")
    document = f"""
    <h4>{html_lib.escape(title)}</h4><div style="overflow-x:auto">
    <table style="border-collapse:collapse; min-width:900px"><thead><tr>{header}</tr></thead>
    <tbody>{''.join(body)}</tbody></table></div>
    <style>table th, table td {{ border:1px solid #bbb; padding:5px 8px; text-align:right; }}
    table th:first-child, table td:first-child {{ text-align:left; }} table thead {{ background:#eee; }}</style>
    """
    display(HTML(document))


def build_thesis_tables(results: dict, repeatability_results: dict) -> dict:
    accuracy_rows, ik_rows, dynamics_rows, per_joint_rows = [], [], [], []
    for name, log in results.items():
        metrics = log["metrics"]
        accuracy_rows.append({
            "motion": name,
            "mean_error_mm": metrics["mean_cartesian_error_mm"],
            "rmse_mm": metrics["rmse_cartesian_error_mm"],
            "max_error_mm": metrics["max_cartesian_error_mm"],
            "reference_length_cm": 100.0 * metrics["reference_path_length_m"],
            "actual_length_cm": 100.0 * metrics["actual_path_length_m"],
            "path_efficiency": metrics["path_efficiency"],
            "wall_control_rate_hz": metrics["wall_control_rate_hz"],
        })
        dynamics_rows.append({
            "motion": name,
            "max_speed_m_s": metrics["max_eef_speed_m_s"],
            "rms_speed_m_s": metrics["rms_eef_speed_m_s"],
            "max_acceleration_m_s2": metrics["max_eef_acceleration_m_s2"],
            "rms_acceleration_m_s2": metrics["rms_eef_acceleration_m_s2"],
            "max_jerk_m_s3": metrics["max_eef_jerk_m_s3"],
            "rms_jerk_m_s3": metrics["rms_eef_jerk_m_s3"],
            "normalized_jerk": metrics["normalized_jerk"],
            "max_joint_velocity_rad_s": metrics["max_joint_velocity_rad_s"],
            "max_joint_acceleration_rad_s2": metrics["max_joint_acceleration_rad_s2"],
            "joint_discontinuity_rad": metrics["joint_discontinuity_rad"],
            "minimum_joint_limit_margin_rad": metrics["minimum_joint_limit_margin_rad"],
            "joint_command_clip_events": metrics["joint_command_clip_events"],
        })
        for joint_index, joint_name in enumerate(ARM_JOINT_NAMES_BARE):
            per_joint_rows.append({
                "motion": name,
                "joint": joint_name,
                "max_velocity_rad_s": metrics["max_joint_velocity_per_joint_rad_s"][joint_index],
                "max_acceleration_rad_s2": metrics["max_joint_acceleration_per_joint_rad_s2"][joint_index],
                "minimum_limit_margin_rad": metrics["minimum_joint_limit_margin_per_joint_rad"][joint_index],
            })
        if log["space"] == "cartesian":
            ik_rows.append({
                "motion": name,
                "strict_success_percent": 100.0 * metrics["solver_success_rate"],
                "non_hold_percent": 100.0 * metrics["ik_non_hold_rate"],
                "mean_position_error_mm": metrics["mean_ik_position_error_mm"],
                "max_position_error_mm": metrics["max_ik_position_error_mm"],
                "mean_rotation_error_deg": np.degrees(metrics["mean_ik_rotation_error_rad"]),
                "max_rotation_error_deg": np.degrees(metrics["max_ik_rotation_error_rad"]),
                "mean_solve_time_ms": metrics["mean_ik_solve_time_ms"],
                "p95_solve_time_ms": metrics["p95_ik_solve_time_ms"],
                "max_solve_time_ms": metrics["max_ik_solve_time_ms"],
                "status_counts": metrics["ik_status_counts"],
            })

    circle = results["circle"]["circle_metrics"]
    circle_rows = [{
        "motion": "circle", "reference_radius_cm": 100.0 * circle["reference_radius_m"],
        "fitted_radius_cm": 100.0 * circle["fitted_actual_radius_m"], "radius_error_mm": circle["radius_error_mm"],
        "radial_rmse_mm": circle["radial_tracking_rmse_mm"], "radial_max_mm": circle["radial_tracking_max_mm"],
        "fit_residual_mm": circle["fitted_circle_radial_rmse_mm"], "planarity_rmse_mm": circle["planarity_rmse_mm"],
        "closure_error_mm": circle["closure_error_mm"],
    }]

    repeatability_rows = []
    for trajectory_name, result in repeatability_results.items():
        repeated = result["metrics"]
        repeatability_rows.append({
            "trajectory": trajectory_name, "n_trials": repeated["n_trials"],
            "trajectory_rmse_mm": repeated["trajectory_repeatability_rmse_mm"],
            "trajectory_max_mm": repeated["trajectory_repeatability_max_mm"],
            "mean_endpoint_deviation_mm": repeated["final_deviation_mean_mm"],
            "std_endpoint_deviation_mm": repeated["final_deviation_std_mm"],
            "max_endpoint_deviation_mm": repeated["final_deviation_max_mm"],
            "mean_final_target_error_mm": repeated["final_target_error_mean_mm"],
            "std_final_target_error_mm": repeated["final_target_error_std_mm"],
        })

    conditions_rows = [
        {"condition": "Command regime", "value": COMMAND_REGIME, "unit": "declared condition"},
        {"condition": "Cartesian orientation policy", "value": CARTESIAN_ORIENTATION_POLICY, "unit": "constraint"},
        {"condition": "MuJoCo physics frequency", "value": SIM_FREQ_HZ, "unit": "Hz"},
        {"condition": "State logging / derivative frequency", "value": STATE_LOGGING_FREQUENCY_HZ, "unit": "Hz"},
        {"condition": "Physics-substep state logging", "value": "disabled", "unit": "boolean"},
        {"condition": "Joint controller frequency", "value": CTRL_FREQ_HZ, "unit": "Hz"},
        {"condition": "Cartesian ticks per waypoint", "value": CTRL_TICKS_PER_CARTESIAN_WAYPOINT, "unit": "ticks"},
        {"condition": "Joint ticks per waypoint", "value": CTRL_TICKS_PER_JOINT_WAYPOINT, "unit": "ticks"},
        {"condition": "Reset settling time", "value": SETTLE_SECONDS, "unit": "s"},
        {"condition": "Major-joint proportional gain", "value": CONTROLLER_CONFIG["kp_major"], "unit": "model units"},
        {"condition": "Wrist proportional gain", "value": CONTROLLER_CONFIG["kp_wrist"], "unit": "model units"},
        {"condition": "Major-joint damping", "value": CONTROLLER_CONFIG["damping_major"], "unit": "model units"},
        {"condition": "Wrist damping", "value": CONTROLLER_CONFIG["damping_wrist"], "unit": "model units"},
        {"condition": "Gravity compensation strength", "value": CONTROLLER_CONFIG["gravity_strength"], "unit": "ratio"},
        {"condition": "Actuator force limits", "value": CONTROLLER_CONFIG["actuator_force_limits"], "unit": "condition"},
        {"condition": "Repeatability geometries", "value": ", ".join(REPEATABILITY_GEOMETRIES), "unit": "list"},
        {"condition": "Repeatability trials per geometry", "value": REPEATABILITY_TRIALS, "unit": "trials"},
    ]
    return {
        "cartesian_accuracy": accuracy_rows, "ik_reliability": ik_rows,
        "dynamics_smoothness": dynamics_rows, "per_joint_limits": per_joint_rows,
        "circle_geometry": circle_rows, "repeatability": repeatability_rows,
        "experimental_conditions": conditions_rows,
    }


THESIS_TABLES = build_thesis_tables(RESULTS, REPEATABILITY_RESULTS)


In [ ]:
display_metric_table(
    "Table 1. Cartesian accuracy, path fidelity, and real-time execution",
    THESIS_TABLES["cartesian_accuracy"],
    [("motion", "Motion", 0), ("mean_error_mm", "Mean error [mm]", 3), ("rmse_mm", "RMSE [mm]", 3),
     ("max_error_mm", "Max error [mm]", 3), ("reference_length_cm", "Reference length [cm]", 2),
     ("actual_length_cm", "Actual length [cm]", 2), ("path_efficiency", "Path efficiency", 4),
     ("wall_control_rate_hz", "Wall control rate [Hz]", 2)],
)
display_metric_table(
    "Table 2. IK reliability and latency distribution", THESIS_TABLES["ik_reliability"],
    [("motion", "Motion", 0), ("strict_success_percent", "Strict success [%]", 1),
     ("non_hold_percent", "Non-hold [%]", 1), ("mean_position_error_mm", "Mean IK position error [mm]", 3),
     ("max_position_error_mm", "Max IK position error [mm]", 3),
     ("mean_rotation_error_deg", "Mean rotation error [deg]", 3), ("max_rotation_error_deg", "Max rotation error [deg]", 3),
     ("mean_solve_time_ms", "Mean solve [ms]", 2), ("p95_solve_time_ms", "p95 solve [ms]", 2),
     ("max_solve_time_ms", "Max solve [ms]", 2), ("status_counts", "IK status counts", 0)],
)
display_metric_table(
    "Table 3. Dynamic, joint, and clipping metrics", THESIS_TABLES["dynamics_smoothness"],
    [("motion", "Motion", 0), ("max_speed_m_s", "Max speed [m/s]", 3), ("rms_speed_m_s", "RMS speed [m/s]", 3),
     ("max_acceleration_m_s2", "Max acceleration [m/s²]", 3), ("rms_acceleration_m_s2", "RMS acceleration [m/s²]", 3),
     ("max_jerk_m_s3", "Max jerk [m/s³]", 2), ("rms_jerk_m_s3", "RMS jerk [m/s³]", 2),
     ("normalized_jerk", "Normalized jerk", 2), ("max_joint_velocity_rad_s", "Max joint velocity [rad/s]", 3),
     ("max_joint_acceleration_rad_s2", "Max joint acceleration [rad/s²]", 3),
     ("joint_discontinuity_rad", "Joint discontinuity [rad]", 4),
     ("minimum_joint_limit_margin_rad", "Min joint margin [rad]", 4),
     ("joint_command_clip_events", "Command clips", 0)],
)
display_metric_table(
    "Table 4. Per-joint velocity, acceleration, and limit margin (appendix)", THESIS_TABLES["per_joint_limits"],
    [("motion", "Motion", 0), ("joint", "Joint", 0), ("max_velocity_rad_s", "Max velocity [rad/s]", 3),
     ("max_acceleration_rad_s2", "Max acceleration [rad/s²]", 3),
     ("minimum_limit_margin_rad", "Minimum limit margin [rad]", 4)],
)
display_metric_table(
    "Table 5. Circle-specific geometry", THESIS_TABLES["circle_geometry"],
    [("motion", "Motion", 0), ("reference_radius_cm", "Reference radius [cm]", 3),
     ("fitted_radius_cm", "Fitted radius [cm]", 3), ("radius_error_mm", "Radius error [mm]", 3),
     ("radial_rmse_mm", "Radial tracking RMSE [mm]", 3), ("radial_max_mm", "Max radial error [mm]", 3),
     ("fit_residual_mm", "Best-fit residual [mm]", 3), ("planarity_rmse_mm", "Planarity RMSE [mm]", 3),
     ("closure_error_mm", "Closure error [mm]", 3)],
)
display_metric_table(
    "Table 6. Endpoint and whole-trajectory repeatability", THESIS_TABLES["repeatability"],
    [("trajectory", "Trajectory", 0), ("n_trials", "N", 0),
     ("trajectory_rmse_mm", "Trajectory dispersion RMSE [mm]", 3), ("trajectory_max_mm", "Max trajectory deviation [mm]", 3),
     ("mean_endpoint_deviation_mm", "Mean endpoint deviation [mm]", 3),
     ("std_endpoint_deviation_mm", "SD endpoint deviation [mm]", 3),
     ("max_endpoint_deviation_mm", "Max endpoint deviation [mm]", 3),
     ("mean_final_target_error_mm", "Mean final target error [mm]", 3),
     ("std_final_target_error_mm", "SD final target error [mm]", 3)],
)
display_metric_table(
    "Table 7. Frozen experimental conditions", THESIS_TABLES["experimental_conditions"],
    [("condition", "Condition", 0), ("value", "Value", 3), ("unit", "Unit", 0)],
)


In [ ]:
def save_rows_csv(rows: list, path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = list(rows[0].keys()) if rows else []
    with path.open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow({key: json.dumps(_json_value(value)) if isinstance(value, (dict, list, tuple, np.ndarray)) else value for key, value in row.items()})
    return path


summary_directory = BENCHMARK_OUTPUT_ROOT / "_summary"
summary_directory.mkdir(parents=True, exist_ok=True)
THESIS_TABLE_PATHS = {
    table_name: save_rows_csv(rows, summary_directory / f"thesis_{table_name}.csv")
    for table_name, rows in THESIS_TABLES.items()
}

reproducibility_manifest = {
    "mujoco_version": getattr(mujoco, "__version__", "unknown"),
    "numpy_version": np.__version__,
    "mjcf_path": str(SURENA_ARM_XML),
    "mjcf_sha256": hashlib.sha256(Path(SURENA_ARM_XML).read_bytes()).hexdigest(),
    "controller_config": CONTROLLER_CONFIG,
    "benchmark_geometry": BENCHMARK_GEOMETRY,
    "command_regime": COMMAND_REGIME,
    "cartesian_orientation_policy": CARTESIAN_ORIENTATION_POLICY,
    "fixed_orientation_quaternion_by_benchmark_wxyz": {
        name: log.get("fixed_orientation_quat_wxyz") for name, log in RESULTS.items() if log["space"] == "cartesian"
    },
    "simulation_frequency_hz": SIM_FREQ_HZ,
    "controller_frequency_hz": CTRL_FREQ_HZ,
    "state_logging_frequency_hz": STATE_LOGGING_FREQUENCY_HZ,
    "physics_substep_logging": False,
    "physics_steps_between_logged_samples": STEPS_PER_CTRL,
    "cartesian_ticks_per_waypoint": CTRL_TICKS_PER_CARTESIAN_WAYPOINT,
    "joint_ticks_per_waypoint": CTRL_TICKS_PER_JOINT_WAYPOINT,
    "repeatability_geometries": REPEATABILITY_GEOMETRIES,
    "repeatability_trials_per_geometry": REPEATABILITY_TRIALS,
}
manifest_path = summary_directory / "thesis_reproducibility_manifest.json"
manifest_path.write_text(json.dumps(_json_value(reproducibility_manifest), indent=2))

print("Thesis tables exported:")
for table_path in THESIS_TABLE_PATHS.values():
    print(" ", table_path)
print(" ", manifest_path)
